In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors

In [2]:
books_features = pd.read_csv("../Data/books_features.csv")

books_features.head()

,book_id,title,authors,original_publication_year,average_rating,ratings_count,image_url,tag_name
0,1,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,2008.0,4.34,4780653,https://images.gr-assets.com/books/1447303603m...,young-adult fiction dystopian dystopia fantasy...
1,2,Harry Potter and the Sorcerer's Stone (Harry P...,"J.K. Rowling, Mary GrandPré",1997.0,4.44,4602479,https://images.gr-assets.com/books/1474154022m...,fantasy young-adult fiction harry-potter ya se...
2,3,"Twilight (Twilight, #1)",Stephenie Meyer,2005.0,3.57,3866839,https://images.gr-assets.com/books/1361039443m...,young-adult fantasy vampires ya fiction parano...
3,4,To Kill a Mockingbird,Harper Lee,1960.0,4.25,3198671,https://images.gr-assets.com/books/1361975680m...,classics classic historical-fiction school clà...
4,5,The Great Gatsby,F. Scott Fitzgerald,1925.0,3.89,2683664,https://images.gr-assets.com/books/1490528560m...,classics fiction classic literature school his...


In [3]:
books_features.shape

(10000, 8)

In [4]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

tfidf_matrix = tfidf.fit_transform(books_features["tag_name"])

In [5]:
tfidf_matrix.shape

(10000, 2877)

In [6]:
content_similarity = cosine_similarity(tfidf_matrix)

In [7]:
content_similarity.shape

(10000, 10000)

In [8]:
title_to_index = pd.Series(
    books_features.index,
    index=books_features["title"]
).to_dict()

In [9]:
title_to_index["The Hunger Games (The Hunger Games, #1)"]

0

In [10]:
def recommend_content(title, n=10):
    
    if title not in title_to_index:
        return pd.DataFrame()
    
    idx = title_to_index[title]
    
    scores = list(enumerate(content_similarity[idx]))
    
    scores = sorted(
        scores,
        key=lambda x: x[1],
        reverse=True
    )
    
    scores = [
        (book_idx, score)
        for book_idx, score in scores
        if book_idx != idx
    ][:n]
    
    recommendations = books_features.iloc[
        [x[0] for x in scores]
    ].copy()
    
    recommendations["similarity"] = [
        x[1] for x in scores
    ]
    
    return recommendations[
        ["book_id", "title", "authors", "similarity"]
    ]

In [11]:
recommend_content(
    "The Hunger Games (The Hunger Games, #1)"
)

,book_id,title,authors,similarity
16,17,"Catching Fire (The Hunger Games, #2)",Suzanne Collins,0.946378
19,20,"Mockingjay (The Hunger Games, #3)",Suzanne Collins,0.935665
506,507,The Hunger Games Trilogy Boxset (The Hunger Ga...,Suzanne Collins,0.896548
11,12,"Divergent (Divergent, #1)",Veronica Roth,0.798836
2300,2301,"The Ask and the Answer (Chaos Walking, #2)",Patrick Ness,0.761434
780,781,"Specials (Uglies, #3)",Scott Westerfeld,0.760736
2562,2563,"Monsters of Men (Chaos Walking, #3)",Patrick Ness,0.757503
279,280,"Delirium (Delirium, #1)",Lauren Oliver,0.748676
518,519,"Pretties (Uglies, #2)",Scott Westerfeld,0.748150
1397,1398,"Extras (Uglies, #4)",Scott Westerfeld,0.741606


In [12]:
%whos

Variable             Type               Data/Info
-------------------------------------------------
NearestNeighbors     ABCMeta            <class 'sklearn.neighbors<...>rvised.NearestNeighbors'>
TfidfVectorizer      type               <class 'sklearn.feature_e<...>on.text.TfidfVectorizer'>
books_features       DataFrame          Shape: (10000, 8)
content_similarity   ndarray            10000x10000: 100000000 elems, type `float64`, 800000000 bytes (762.939453125 Mb)
cosine_similarity    function           <function cosine_similari<...>ty at 0x000001F9752DE5C0>
np                   module             <module 'numpy' from 'c:\<...>ges\\numpy\\__init__.py'>
pd                   module             <module 'pandas' from 'c:<...>es\\pandas\\__init__.py'>
recommend_content    function           <function recommend_conte<...>nt at 0x000001F9757B82C0>
tfidf                TfidfVectorizer    TfidfVectorizer(max_featu<...>00, stop_words='english')
tfidf_matrix         csr_matrix         <Compress

In [13]:
[x for x in dir() if "collab" in x.lower() or "hybrid" in x.lower() or "user" in x.lower()]

[]

In [14]:
ratings = pd.read_csv("../Data/ratings.csv")

ratings.head()

,user_id,book_id,rating
0,1,258,5
1,2,4081,4
2,2,260,5
3,2,9296,5
4,2,2318,3


In [15]:
ratings.shape

(5976479, 3)

In [16]:
from scipy.sparse import csr_matrix

user_ids = ratings["user_id"].astype("category")
book_ids = ratings["book_id"].astype("category")

user_item_matrix = csr_matrix(
    (
        ratings["rating"].values,
        (user_ids.cat.codes, book_ids.cat.codes)
    )
)

user_item_matrix.shape

(53424, 10000)

In [17]:
collaborative_model = NearestNeighbors(
    n_neighbors=11,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

collaborative_model.fit(user_item_matrix.T)

print("Collaborative model fitted.")

Collaborative model fitted.


In [18]:
book_id = 1

book_index = book_ids.cat.categories.get_loc(book_id)

distances, indices = collaborative_model.kneighbors(
    user_item_matrix.T[book_index],
    n_neighbors=11
)

distances

array([[0.        , 0.27698878, 0.31265304, 0.4101597 , 0.4388581 ,
        0.47668349, 0.54099195, 0.54144194, 0.54995368, 0.55130358,
        0.55166268]])

In [19]:
similarities = 1 - distances[0]

similarities

array([1.        , 0.72301122, 0.68734696, 0.5898403 , 0.5611419 ,
       0.52331651, 0.45900805, 0.45855806, 0.45004632, 0.44869642,
       0.44833732])

In [20]:
def recommend_collaborative(book_id, n=10):

    # Find the position of the book
    book_index = book_ids.cat.categories.get_loc(book_id)

    # Find similar books
    distances, indices = collaborative_model.kneighbors(
        user_item_matrix.T[book_index],
        n_neighbors=n + 1
    )

    similarities = 1 - distances[0]

    # Remove the original book
    similar_indices = indices[0][1:]
    similar_scores = similarities[1:]

    # Convert matrix indices back to book IDs
    similar_book_ids = book_ids.cat.categories[similar_indices]

    recommendations = books_features[
        books_features["book_id"].isin(similar_book_ids)
    ].copy()

    # Preserve recommendation order
    score_map = dict(zip(similar_book_ids, similar_scores))

    recommendations["similarity"] = recommendations["book_id"].map(score_map)

    recommendations = recommendations.sort_values(
        "similarity",
        ascending=False
    )

    return recommendations[
        ["book_id", "title", "authors", "similarity"]
    ].head(n)

In [21]:
recommend_collaborative(1)

,book_id,title,authors,similarity
16,17,"Catching Fire (The Hunger Games, #2)",Suzanne Collins,0.723011
19,20,"Mockingjay (The Hunger Games, #3)",Suzanne Collins,0.687347
1,2,Harry Potter and the Sorcerer's Stone (Harry P...,"J.K. Rowling, Mary GrandPré",0.589840
2,3,"Twilight (Twilight, #1)",Stephenie Meyer,0.561142
11,12,"Divergent (Divergent, #1)",Veronica Roth,0.523317
30,31,The Help,Kathryn Stockett,0.459008
15,16,"The Girl with the Dragon Tattoo (Millennium, #1)","Stieg Larsson, Reg Keeland",0.458558
20,21,Harry Potter and the Order of the Phoenix (Har...,"J.K. Rowling, Mary GrandPré",0.450046
3,4,To Kill a Mockingbird,Harper Lee,0.448696
24,25,Harry Potter and the Deathly Hallows (Harry Po...,"J.K. Rowling, Mary GrandPré",0.448337


## 3.3 Hybrid Recommendation System

The hybrid recommender combines content-based and collaborative
similarity scores.

Hybrid Score = α × Content Score + (1 − α) × Collaborative Score

where α controls the importance given to the content-based model.

In [22]:
%whos

Variable                  Type                Data/Info
-------------------------------------------------------
NearestNeighbors          ABCMeta             <class 'sklearn.neighbors<...>rvised.NearestNeighbors'>
TfidfVectorizer           type                <class 'sklearn.feature_e<...>on.text.TfidfVectorizer'>
book_id                   int                 1
book_ids                  Series              Shape: (5976479,)
book_index                int                 0
books_features            DataFrame           Shape: (10000, 8)
collaborative_model       NearestNeighbors    NearestNeighbors(algorith<...>_jobs=-1, n_neighbors=11)
content_similarity        ndarray             10000x10000: 100000000 elems, type `float64`, 800000000 bytes (762.939453125 Mb)
cosine_similarity         function            <function cosine_similari<...>ty at 0x000001F9752DE5C0>
csr_matrix                ABCMeta             <class 'scipy.sparse._csr.csr_matrix'>
distances                 ndarray           

In [23]:
# Final BookNest model configuration

FINAL_ALPHA = 0.3
FINAL_K = 10

print("Final content weight:", FINAL_ALPHA)
print("Final collaborative weight:", 1 - FINAL_ALPHA)
print("Final recommendation count:", FINAL_K)

Final content weight: 0.3
Final collaborative weight: 0.7
Final recommendation count: 10


In [24]:
def recommend_hybrid(title, alpha=FINAL_ALPHA, n=FINAL_K):
    
    # Check whether the book exists
    if title not in title_to_index:
        return pd.DataFrame()
    
    # Get the book's index
    content_idx = title_to_index[title]
    
    # Get the actual book_id
    book_id = books_features.iloc[content_idx]["book_id"]
    
    # -----------------------------
    # CONTENT-BASED SCORES
    # -----------------------------
    
    content_scores = content_similarity[content_idx]
    
    # -----------------------------
    # COLLABORATIVE SCORES
    # -----------------------------
    
    book_index = book_ids.cat.categories.get_loc(book_id)
    
    distances, indices = collaborative_model.kneighbors(
        user_item_matrix.T[book_index],
        n_neighbors=n + 1
    )
    
    collaborative_scores = np.zeros(len(books_features))
    
    similar_indices = indices[0][1:]
    similar_scores = 1 - distances[0][1:]
    
    similar_book_ids = book_ids.cat.categories[similar_indices]
    
    for similar_book_id, score in zip(similar_book_ids, similar_scores):
        
        matching_rows = books_features.index[
            books_features["book_id"] == similar_book_id
        ]
        
        for row in matching_rows:
            collaborative_scores[row] = score
    
    # -----------------------------
    # HYBRID SCORE
    # -----------------------------
    
    hybrid_scores = (
        alpha * content_scores
        + (1 - alpha) * collaborative_scores
    )
    
    # Remove the original book
    hybrid_scores[content_idx] = -1
    
    # Get top recommendations
    top_indices = np.argsort(
        hybrid_scores
    )[::-1][:n]
    
    recommendations = books_features.iloc[top_indices].copy()
    
    recommendations["content_score"] = content_scores[top_indices]
    recommendations["collaborative_score"] = collaborative_scores[top_indices]
    recommendations["hybrid_score"] = hybrid_scores[top_indices]
    
    return recommendations[
        [
            "book_id",
            "title",
            "authors",
            "content_score",
            "collaborative_score",
            "hybrid_score"
        ]
    ]

In [25]:
recommend_hybrid(
    "The Hunger Games (The Hunger Games, #1)"
)

,book_id,title,authors,content_score,collaborative_score,hybrid_score
16,17,"Catching Fire (The Hunger Games, #2)",Suzanne Collins,0.946378,0.723011,0.790021
19,20,"Mockingjay (The Hunger Games, #3)",Suzanne Collins,0.935665,0.687347,0.761842
11,12,"Divergent (Divergent, #1)",Veronica Roth,0.798836,0.523317,0.605972
1,2,Harry Potter and the Sorcerer's Stone (Harry P...,"J.K. Rowling, Mary GrandPré",0.381345,0.589840,0.527292
2,3,"Twilight (Twilight, #1)",Stephenie Meyer,0.439911,0.561142,0.524772
20,21,Harry Potter and the Order of the Phoenix (Har...,"J.K. Rowling, Mary GrandPré",0.396281,0.450046,0.433917
24,25,Harry Potter and the Deathly Hallows (Harry Po...,"J.K. Rowling, Mary GrandPré",0.378365,0.448337,0.427346
30,31,The Help,Kathryn Stockett,0.223314,0.459008,0.388300
15,16,"The Girl with the Dragon Tattoo (Millennium, #1)","Stieg Larsson, Reg Keeland",0.191727,0.458558,0.378509
3,4,To Kill a Mockingbird,Harper Lee,0.190870,0.448696,0.371349


In [26]:
import os

print(os.getcwd())

c:\Users\Yashi Sharma\Desktop\BookNest\Notebooks


In [27]:
import sys
sys.path.append("..")

from src.content_recommender import recommend_content

In [28]:
recommend_content(
    "The Hunger Games (The Hunger Games, #1)"
)

,book_id,title,authors,similarity
16,17,"Catching Fire (The Hunger Games, #2)",Suzanne Collins,0.946378
19,20,"Mockingjay (The Hunger Games, #3)",Suzanne Collins,0.935665
506,507,The Hunger Games Trilogy Boxset (The Hunger Ga...,Suzanne Collins,0.896548
11,12,"Divergent (Divergent, #1)",Veronica Roth,0.798836
2300,2301,"The Ask and the Answer (Chaos Walking, #2)",Patrick Ness,0.761434
780,781,"Specials (Uglies, #3)",Scott Westerfeld,0.760736
2562,2563,"Monsters of Men (Chaos Walking, #3)",Patrick Ness,0.757503
279,280,"Delirium (Delirium, #1)",Lauren Oliver,0.748676
518,519,"Pretties (Uglies, #2)",Scott Westerfeld,0.748150
1397,1398,"Extras (Uglies, #4)",Scott Westerfeld,0.741606


In [29]:
import sys
sys.path.append("..")

from src.collaborative_recommender import recommend_collaborative

recommend_collaborative(1)

,book_id,title,authors,similarity
16,17,"Catching Fire (The Hunger Games, #2)",Suzanne Collins,0.723011
19,20,"Mockingjay (The Hunger Games, #3)",Suzanne Collins,0.687347
1,2,Harry Potter and the Sorcerer's Stone (Harry P...,"J.K. Rowling, Mary GrandPré",0.589840
2,3,"Twilight (Twilight, #1)",Stephenie Meyer,0.561142
11,12,"Divergent (Divergent, #1)",Veronica Roth,0.523317
30,31,The Help,Kathryn Stockett,0.459008
15,16,"The Girl with the Dragon Tattoo (Millennium, #1)","Stieg Larsson, Reg Keeland",0.458558
20,21,Harry Potter and the Order of the Phoenix (Har...,"J.K. Rowling, Mary GrandPré",0.450046
3,4,To Kill a Mockingbird,Harper Lee,0.448696
24,25,Harry Potter and the Deathly Hallows (Harry Po...,"J.K. Rowling, Mary GrandPré",0.448337


In [30]:
import sys
sys.path.append("..")

from src.hybrid_recommender import recommend_hybrid

recommend_hybrid(
    "The Hunger Games (The Hunger Games, #1)"
)

,book_id,title,authors,content_score,collaborative_score,hybrid_score
16,17,"Catching Fire (The Hunger Games, #2)",Suzanne Collins,0.946378,0.723011,0.790021
19,20,"Mockingjay (The Hunger Games, #3)",Suzanne Collins,0.935665,0.687347,0.761842
11,12,"Divergent (Divergent, #1)",Veronica Roth,0.798836,0.523317,0.605972
1,2,Harry Potter and the Sorcerer's Stone (Harry P...,"J.K. Rowling, Mary GrandPré",0.381345,0.589840,0.527292
2,3,"Twilight (Twilight, #1)",Stephenie Meyer,0.439911,0.561142,0.524772
20,21,Harry Potter and the Order of the Phoenix (Har...,"J.K. Rowling, Mary GrandPré",0.396281,0.450046,0.433917
24,25,Harry Potter and the Deathly Hallows (Harry Po...,"J.K. Rowling, Mary GrandPré",0.378365,0.448337,0.427346
30,31,The Help,Kathryn Stockett,0.223314,0.459008,0.388300
15,16,"The Girl with the Dragon Tattoo (Millennium, #1)","Stieg Larsson, Reg Keeland",0.191727,0.458558,0.378509
3,4,To Kill a Mockingbird,Harper Lee,0.190870,0.448696,0.371349


In [31]:
import sys
sys.path.append("..")

from src.content_recommender import recommend_content
from src.collaborative_recommender import recommend_collaborative
from src.hybrid_recommender import recommend_hybrid

print("Content-based:")
print(recommend_content(
    "The Hunger Games (The Hunger Games, #1)"
).head(3))

print("\nCollaborative:")
print(recommend_collaborative(1).head(3))

print("\nHybrid:")
print(recommend_hybrid(
    "The Hunger Games (The Hunger Games, #1)"
).head(3))

Content-based:
     book_id                                              title  \
16        17               Catching Fire (The Hunger Games, #2)   
19        20                  Mockingjay (The Hunger Games, #3)   
506      507  The Hunger Games Trilogy Boxset (The Hunger Ga...   

             authors  similarity  
16   Suzanne Collins    0.946378  
19   Suzanne Collins    0.935665  
506  Suzanne Collins    0.896548  

Collaborative:
    book_id                                              title  \
16       17               Catching Fire (The Hunger Games, #2)   
19       20                  Mockingjay (The Hunger Games, #3)   
1         2  Harry Potter and the Sorcerer's Stone (Harry P...   

                        authors  similarity  
16              Suzanne Collins    0.723011  
19              Suzanne Collins    0.687347  
1   J.K. Rowling, Mary GrandPré    0.589840  

Hybrid:
    book_id                                 title          authors  \
16       17  Catching Fire (The 

In [32]:
import sys
sys.path.append("..")

from src.content_recommender import recommend_content
from src.collaborative_recommender import recommend_collaborative
from src.hybrid_recommender import recommend_hybrid

print("Modules imported successfully.")

Modules imported successfully.


In [33]:
print("Content:")
print(recommend_content(
    "The Hunger Games (The Hunger Games, #1)"
).head(3))

print("\nCollaborative:")
print(recommend_collaborative(1).head(3))

print("\nHybrid:")
print(recommend_hybrid(
    "The Hunger Games (The Hunger Games, #1)"
).head(3))

Content:
     book_id                                              title  \
16        17               Catching Fire (The Hunger Games, #2)   
19        20                  Mockingjay (The Hunger Games, #3)   
506      507  The Hunger Games Trilogy Boxset (The Hunger Ga...   

             authors  similarity  
16   Suzanne Collins    0.946378  
19   Suzanne Collins    0.935665  
506  Suzanne Collins    0.896548  

Collaborative:
    book_id                                              title  \
16       17               Catching Fire (The Hunger Games, #2)   
19       20                  Mockingjay (The Hunger Games, #3)   
1         2  Harry Potter and the Sorcerer's Stone (Harry P...   

                        authors  similarity  
16              Suzanne Collins    0.723011  
19              Suzanne Collins    0.687347  
1   J.K. Rowling, Mary GrandPré    0.589840  

Hybrid:
    book_id                                 title          authors  \
16       17  Catching Fire (The Hunger